# Módulo 03 · Aula 01 — Fundamentos Relacionais

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

## A dor da Aurora

> *"Os dados estão em 14 planilhas diferentes. A do comercial tem 'Campinas', a do financeiro tem 'campinas/SP', a do estoque tem 'CPS'. Quando alguém corrige o preço de um produto, corrige em uma planilha só. Ontem descobrimos que o mesmo cliente aparece 4 vezes com e-mails diferentes."*
> — Diretora de Operações

Seu CSV do Módulo 01 funciona. Mas ele é **um arquivo plano**, e arquivos planos têm limites que aparecem rápido.

## O que você vai aprender aqui

| # | Tópico | Por que importa |
|---|--------|-----------------|
| 1 | O problema do arquivo plano | Anomalias de inserção, atualização e remoção |
| 2 | Normalização | Cada fato em um lugar só |
| 3 | SQLite e conexão via Python | Banco sem servidor, roda no notebook |
| 4 | Tipos de dados (e a afinidade do SQLite) | ⚠️ A pegadinha número 1 |
| 5 | `CREATE TABLE` e DDL | Definir a estrutura |
| 6 | Chave primária (PK) | Identidade das linhas |
| 7 | Chave estrangeira (FK) | Integridade referencial |
| 8 | Constraints | O banco defendendo seus dados |
| 9 | Diagrama ER | Desenhar antes de codificar |

## 1. Por que não continuar com planilhas

Olhe esta "planilha" da Aurora. Ela é exatamente o CSV do Módulo 01:

| pedido | data | cliente | email | cidade | uf | produto | categoria | preço | qtd |
|--------|------|---------|-------|--------|----|---------|-----------|-------|-----|
| 1001 | 2026-07-01 | Maria Souza | maria@x.com | Campinas | SP | Notebook Dell | Informática | 2599.90 | 2 |
| 1002 | 2026-07-01 | João Lima | joao@x.com | São Paulo | SP | Mouse Logitech | Periféricos | 89.90 | 10 |
| 1003 | 2026-07-02 | Maria Souza | maria@x.com | Campinas | SP | Notebook Dell | Informática | 2599.90 | 1 |
| 1004 | 2026-07-03 | Maria Souza | maria@x.com | campinas | SP | Notebook Dell | informática | 2699.90 | 1 |

Repare no que já está errado na linha 4: `campinas` minúsculo, `informática` minúsculo, e o preço mudou. Foi correção ou erro de digitação? **Ninguém sabe.**

### As três anomalias clássicas

| Anomalia | O que é | Exemplo na Aurora |
|----------|---------|-------------------|
| **De atualização** | Mudar um fato exige alterar N linhas | O Notebook Dell subiu para R$ 2.799. Você precisa achar e corrigir **todas** as linhas dele. Esquecer uma = inconsistência permanente. |
| **De inserção** | Não dá para registrar um fato sem outro | Chegou um produto novo no catálogo. Onde você cadastra? Não dá — só existem linhas de **pedido**. |
| **De remoção** | Apagar um fato apaga outro junto | Cancelou o único pedido de Sorocaba? Perdeu o registro de que Sorocaba existe. |

Some a isso: **redundância** (o e-mail da Maria repetido em cada linha), **sem tipos** (o CSV não impede `"dez"` na coluna quantidade), **sem regras** (nada impede quantidade negativa) e **sem concorrência** (duas pessoas salvando o arquivo ao mesmo tempo = uma perde o trabalho).

### A solução: uma tabela para cada *coisa*

O modelo relacional resolve isso com uma regra simples:

> **Cada fato deve ser armazenado em exatamente um lugar.**

Em vez de uma planilha gigante, separamos por **entidade** — cada substantivo do negócio vira uma tabela:

```
   CLIENTE                    PEDIDO                    PRODUTO
 ┌──────────┐              ┌──────────┐              ┌──────────┐
 │ id       │◀────────┐    │ id       │      ┌──────▶│ id       │
 │ nome     │         └────│cliente_id│      │       │ sku      │
 │ email    │              │ data     │      │       │ nome     │
 │ cidade   │              │ status   │      │       │ preço    │
 └──────────┘              └──────────┘      │       └──────────┘
                                 ▲           │            ▲
                                 │           │            │
                            ┌────┴───────────┴────┐       │
                            │   ITEM_PEDIDO       │       │
                            │ pedido_id ──────────┘       │
                            │ produto_id ─────────────────┘
                            │ quantidade                  │
                            │ preco_unitario              │
                            └─────────────────────────────┘
```

Agora:

- O e-mail da Maria existe **uma vez**. Mudou? Um `UPDATE`, uma linha.
- Produto novo entra no catálogo **sem precisar de pedido**.
- Cancelar um pedido não apaga o cliente nem o produto.
- O banco **impede** quantidade negativa, e-mail duplicado, pedido de cliente inexistente.

### Normalização — a intuição

Você vai ouvir falar em "formas normais". A teoria completa é longa, mas 95% do valor está nestas três ideias:

| Forma | Regra | Tradução prática |
|-------|-------|------------------|
| **1FN** | Cada célula tem um valor **atômico**; sem grupos repetidos | Nada de `"mouse, teclado, monitor"` numa célula só, nem colunas `produto1`, `produto2`, `produto3` |
| **2FN** | 1FN + todo atributo depende da chave **inteira** | Numa tabela com chave (pedido, produto), o `nome_do_cliente` não pertence ali — ele depende só do pedido |
| **3FN** | 2FN + nenhum atributo depende de outro **não-chave** | Se `uf` é determinada por `cidade`, e `cidade` não é chave, então `uf` está no lugar errado |

> 🧭 **A regra prática que resolve quase tudo:** se você percebe que está **copiando e colando o mesmo valor** em várias linhas, aquilo provavelmente deveria ser uma tabela separada.

> ⚠️ **Normalizar demais também é problema.** Um banco com 40 tabelas exige 12 JOINs para responder uma pergunta simples. No Módulo 10 você vai ver o caminho oposto (**desnormalização**) para análise. Regra: normalize o banco que **grava** (OLTP); desnormalize o que **lê para análise** (OLAP).

## 2. SQLite — o banco que roda no notebook

Para aprender SQL, o SQLite é ideal:

| | SQLite | PostgreSQL / MySQL |
|---|--------|--------------------|
| Instalação | ✅ Já vem com o Python | Servidor separado |
| Banco é | Um arquivo `.db` | Um processo rodando |
| Concorrência | Limitada (um escritor por vez) | Alta |
| Onde brilha | Apps locais, mobile, testes, protótipos | Produção multiusuário |

**O SQLite não é um brinquedo.** Ele está em todo celular Android e iOS, em navegadores, no Firefox, no Photoshop. É provavelmente o banco de dados mais usado do mundo. Mas ele é **embutido**: não serve para uma API com 500 usuários simultâneos — e é exatamente por isso que no Módulo 05 migramos para PostgreSQL.

O SQL que você aprender aqui é ~90% transferível.

### ⚙️ Preparando o ambiente

O módulo `sqlite3` já vem na biblioteca padrão do Python — nada a instalar.

A célula abaixo cria duas funções que usaremos em todo o módulo:

- **`sql(consulta)`** — executa e mostra o resultado numa tabela formatada
- **`ddl(script)`** — executa comandos de definição (vários de uma vez)

In [ ]:
import sqlite3
from pathlib import Path

# ── Conexão ──────────────────────────────────────────────────
# ":memory:" cria o banco na RAM: some quando o kernel reinicia.
# Perfeito para aula. Para persistir, troque por "aurora.db".
CONN = sqlite3.connect(":memory:")

# ⚠️ CRÍTICO: o SQLite vem com chaves estrangeiras DESLIGADAS por
#    compatibilidade histórica. Sem esta linha, o banco aceita
#    pedidos de clientes que não existem. Ligue SEMPRE.
CONN.execute("PRAGMA foreign_keys = ON")


def _fmt(valor):
    if valor is None:
        return "NULL"
    if isinstance(valor, bool):
        return str(valor)
    if isinstance(valor, float):
        return f"{valor:,.2f}"
    if isinstance(valor, int):
        return f"{valor:,}"
    return str(valor)


def sql(consulta, parametros=(), limite=30):
    """Executa uma consulta e imprime o resultado formatado."""
    try:
        cursor = CONN.execute(consulta, parametros)
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")
        return None

    if cursor.description is None:          # INSERT/UPDATE/DELETE/DDL
        CONN.commit()
        # rowcount == -1 significa "comando que não afeta linhas" (DDL)
        if cursor.rowcount >= 0:
            print(f"✅ OK — {cursor.rowcount} linha(s) afetada(s)")
        else:
            print("✅ OK")
        return None

    colunas = [d[0] for d in cursor.description]
    linhas = cursor.fetchall()
    total = len(linhas)
    linhas = linhas[:limite]

    if not linhas:
        print("(nenhuma linha)")
        return []

    texto = [[_fmt(v) for v in linha] for linha in linhas]
    numerica = [
        any(isinstance(l[i], (int, float)) for l in linhas)
        and all(isinstance(l[i], (int, float)) or l[i] is None for l in linhas)
        for i in range(len(colunas))
    ]
    larguras = [
        max(len(colunas[i]), max(len(l[i]) for l in texto))
        for i in range(len(colunas))
    ]

    def borda(esq, meio, dir_):
        return esq + meio.join("─" * (w + 2) for w in larguras) + dir_

    print(borda("┌", "┬", "┐"))
    print("│ " + " │ ".join(c.ljust(w) for c, w in zip(colunas, larguras)) + " │")
    print(borda("├", "┼", "┤"))
    for linha in texto:
        print("│ " + " │ ".join(
            (v.rjust(w) if numerica[i] else v.ljust(w))
            for i, (v, w) in enumerate(zip(linha, larguras))
        ) + " │")
    print(borda("└", "┴", "┘"))
    rodape = f"{total} linha(s)"
    if total > limite:
        rodape += f" — exibindo as {limite} primeiras"
    print(rodape)
    return linhas


def ddl(script):
    """Executa um script com vários comandos (CREATE, DROP, ALTER...)."""
    try:
        CONN.executescript(script)
        CONN.commit()
        print("✅ Script executado")
    except sqlite3.Error as erro:
        print(f"❌ {type(erro).__name__}: {erro}")


print("Versão do SQLite:", sqlite3.sqlite_version)
print("✅ Funções prontas: sql(...) e ddl(...)")

> 💡 **Prefere escrever SQL puro, sem o `sql(...)` em volta?** Existe a extensão **JupySQL**, que habilita células `%%sql`:
>
> ```
> !pip install jupysql --quiet
> %load_ext sql
> %sql sqlite:///aurora.db
> ```
>
> Depois, uma célula inteira vira SQL:
>
> ```
> %%sql
> SELECT * FROM produtos LIMIT 5
> ```
>
> É mais elegante, mas depende de instalação externa. Neste manual usamos o `sql(...)` para garantir que **tudo funcione em qualquer máquina, sem instalar nada**. Se você habilitar o JupySQL, todo o SQL destas aulas funciona igual — só muda o invólucro.

## 3. Tipos de dados no SQLite

O SQL padrão tem dezenas de tipos. O SQLite tem **cinco classes de armazenamento**:

| Classe | O que guarda |
|--------|--------------|
| `NULL` | Ausência de valor |
| `INTEGER` | Inteiro (1 a 8 bytes) |
| `REAL` | Ponto flutuante (8 bytes) |
| `TEXT` | Texto (UTF-8) |
| `BLOB` | Bytes crus |

### ⚠️ A pegadinha: afinidade de tipo

O SQLite tem **tipagem dinâmica**. Ele *aceita* o tipo declarado como uma "preferência" (afinidade), mas **não impede** você de gravar texto numa coluna `INTEGER`.

Isso é diferente de todos os outros bancos e é a fonte de muita confusão. Veja:

In [ ]:
sql("CREATE TABLE teste_tipos (numero INTEGER, texto TEXT)")
sql("INSERT INTO teste_tipos VALUES ('abc', 123)")   # tipos trocados!
sql("SELECT numero, typeof(numero), texto, typeof(texto) FROM teste_tipos")

> 😬 Gravou `'abc'` numa coluna `INTEGER` sem reclamar. No PostgreSQL isso daria erro imediato.
>
> **Como se proteger:** use `CHECK` constraints (que veremos adiante) ou, a partir do SQLite 3.37, declare a tabela como `STRICT`.

In [ ]:
sql("DROP TABLE teste_tipos")

# STRICT: o SQLite passa a se comportar como um banco de tipagem forte
sql("CREATE TABLE teste_strict (numero INTEGER, texto TEXT) STRICT")
sql("INSERT INTO teste_strict VALUES ('abc', 123)")     # agora dá erro
sql("INSERT INTO teste_strict VALUES (42, 'ok')")       # este passa
sql("SELECT * FROM teste_strict")
sql("DROP TABLE teste_strict")

### Tipos que o SQLite não tem

| Tipo esperado | Como o SQLite guarda | Convenção |
|---------------|----------------------|-----------|
| `BOOLEAN` | `INTEGER` | `0` = falso, `1` = verdadeiro, com `CHECK (col IN (0,1))` |
| `DATE` | `TEXT` | ISO 8601: `'2026-08-12'` — ordena corretamente como texto! |
| `DATETIME` | `TEXT` | `'2026-08-12 14:30:00'` |
| `DECIMAL` | `REAL` | ⚠️ Para dinheiro, considere guardar **centavos como INTEGER** |

> 💰 **Sobre dinheiro:** `REAL` é ponto flutuante e sofre do mesmo problema que você viu na aula 01_01 (`0.1 + 0.2 != 0.3`). Para um relatório está ok. Para um sistema financeiro, guarde centavos em `INTEGER` (`259990` em vez de `2599.90`) ou use `TEXT` com `DECIMAL`. No Módulo 05, o PostgreSQL nos dá o tipo `NUMERIC`, que resolve isso de verdade.

In [ ]:
# Datas em ISO 8601 ordenam corretamente como texto — por isso o formato importa
sql("CREATE TABLE teste_datas (evento TEXT, quando TEXT)")
sql("""INSERT INTO teste_datas VALUES
    ('meio',   '2026-07-15'),
    ('último', '2026-12-01'),
    ('primeiro','2026-01-30')""")
sql("SELECT * FROM teste_datas ORDER BY quando")
sql("DROP TABLE teste_datas")

## 4. `CREATE TABLE` — definindo a estrutura

```sql
CREATE TABLE nome_da_tabela (
    coluna1  TIPO  [constraints],
    coluna2  TIPO  [constraints],
    ...
    [constraints de tabela]
);
```

Vamos construir o schema da Aurora **uma tabela por vez**, entendendo cada decisão.

In [ ]:
# A tabela mais simples do modelo: categorias de produto
ddl("""
CREATE TABLE categorias (
    id           INTEGER PRIMARY KEY,
    nome         TEXT    NOT NULL UNIQUE,
    margem_alvo  REAL    NOT NULL DEFAULT 0.25
);
""")

sql("INSERT INTO categorias (nome, margem_alvo) VALUES ('Notebooks', 0.18)")
sql("INSERT INTO categorias (nome, margem_alvo) VALUES ('Monitores', 0.22)")
sql("INSERT INTO categorias (nome) VALUES ('Periféricos')")   # usa o DEFAULT
sql("SELECT * FROM categorias")

## 5. Chave primária (PRIMARY KEY)

A PK **identifica unicamente** cada linha. Ela é:

- **Única** — não há duas linhas com o mesmo valor
- **Não nula** — sempre preenchida
- **Estável** — nunca muda depois de criada

### Chave natural vs chave artificial

| | Natural | Artificial (*surrogate*) |
|---|---------|--------------------------|
| Exemplo | CPF, e-mail, SKU | `id INTEGER PRIMARY KEY` |
| Vantagem | Já tem significado | Nunca muda, sempre pequena |
| Problema | **Muda.** O cliente troca de e-mail. O SKU é reorganizado. E aí você atualiza a chave em 8 tabelas. | Não significa nada sozinha |

> ✅ **Recomendação prática:** use uma **chave artificial** (`id INTEGER PRIMARY KEY`) como PK, e proteja as chaves naturais com `UNIQUE`. Você ganha os dois: identidade estável **e** integridade dos dados de negócio.

### O `rowid` do SQLite

No SQLite, `id INTEGER PRIMARY KEY` (exatamente essas palavras) é um apelido para o `rowid` interno da tabela — e **se autoincrementa** sem que você peça.

In [ ]:
# Repare: os ids foram gerados sozinhos, na ordem de inserção.
sql("SELECT id, nome FROM categorias ORDER BY id")

> ⚠️ **Por que o `ORDER BY id` acima?** Porque **sem `ORDER BY` não existe ordem garantida.** O banco devolve as linhas na ordem que for mais barata para ele — e isso muda conforme os índices disponíveis e o volume de dados.
>
> Experimente tirar o `ORDER BY` da célula acima: como `nome` é `UNIQUE`, o SQLite tem um índice por nome e pode usá-lo para varrer a tabela, devolvendo em ordem alfabética. Você veria `2, 1, 3`.
>
> **Nunca dependa da "ordem natural" de uma tabela.** Se a ordem importa, diga.

> 💡 **`AUTOINCREMENT` — quase sempre desnecessário.** Você vai ver `id INTEGER PRIMARY KEY AUTOINCREMENT` em muitos tutoriais. No SQLite, isso apenas garante que um id **nunca será reutilizado** após uma remoção, ao custo de uma tabela interna extra e desempenho pior. Sem `AUTOINCREMENT`, se você apagar a última linha, o próximo insert pode reaproveitar aquele id. Para a maioria dos casos, tanto faz. A [própria documentação do SQLite](https://sqlite.org/autoinc.html) recomenda **não usar** salvo necessidade específica.

In [ ]:
# Chave primária COMPOSTA: a identidade vem da combinação de colunas
ddl("""
CREATE TABLE estoque_por_deposito (
    produto_id  INTEGER NOT NULL,
    deposito    TEXT    NOT NULL,
    quantidade  INTEGER NOT NULL DEFAULT 0,
    PRIMARY KEY (produto_id, deposito)
);
""")

sql("INSERT INTO estoque_por_deposito VALUES (1, 'Campinas', 40)")
sql("INSERT INTO estoque_por_deposito VALUES (1, 'São Paulo', 12)")   # ok: depósito diferente
sql("INSERT INTO estoque_por_deposito VALUES (1, 'Campinas', 99)")    # ❌ duplicata
sql("SELECT * FROM estoque_por_deposito")

## 6. Constraints — o banco defendendo seus dados

Uma **constraint** é uma regra que o banco **garante**. Não é validação da aplicação: é impossível violá-la, venha o dado de onde vier.

| Constraint | Garante |
|------------|---------|
| `NOT NULL` | A coluna sempre tem valor |
| `UNIQUE` | Não há valores repetidos |
| `PRIMARY KEY` | `NOT NULL` + `UNIQUE` + identidade |
| `DEFAULT valor` | Valor automático quando omitido |
| `CHECK (condição)` | Regra de negócio arbitrária |
| `REFERENCES` | Integridade referencial (FK) |

> 🛡️ **Por que confiar no banco e não só na aplicação?** Porque os dados vão entrar por vários caminhos: sua API, um script de carga, o console de alguém às 23h de uma sexta-feira, uma integração de terceiro. A aplicação pode ter bug; a constraint não. **O banco é a última linha de defesa e ela nunca dorme.**

In [ ]:
ddl("""
CREATE TABLE produtos (
    id           INTEGER PRIMARY KEY,
    sku          TEXT    NOT NULL UNIQUE,
    nome         TEXT    NOT NULL,
    categoria_id INTEGER NOT NULL,
    preco        REAL    NOT NULL CHECK (preco >= 0),
    custo        REAL    NOT NULL CHECK (custo >= 0),
    estoque      INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0),
    ativo        INTEGER NOT NULL DEFAULT 1 CHECK (ativo IN (0, 1)),

    -- constraint de TABELA: envolve mais de uma coluna
    CHECK (preco >= custo)
);
""")

sql("""INSERT INTO produtos (sku, nome, categoria_id, preco, custo, estoque)
       VALUES ('NB-DELL-15', 'Notebook Dell Inspiron 15', 1, 2599.90, 2120.00, 14)""")
sql("SELECT * FROM produtos")

In [ ]:
# Cada constraint sendo testada
print("① Preço negativo:")
sql("INSERT INTO produtos (sku,nome,categoria_id,preco,custo) VALUES ('X1','Teste',1,-10,5)")

print("\n② SKU duplicado:")
sql("INSERT INTO produtos (sku,nome,categoria_id,preco,custo) VALUES ('NB-DELL-15','Outro',1,100,50)")

print("\n③ Nome nulo:")
sql("INSERT INTO produtos (sku,nome,categoria_id,preco,custo) VALUES ('X2',NULL,1,100,50)")

print("\n④ 'ativo' fora do domínio:")
sql("INSERT INTO produtos (sku,nome,categoria_id,preco,custo,ativo) VALUES ('X3','Teste',1,100,50,7)")

print("\n⑤ Preço abaixo do custo (constraint de tabela):")
sql("INSERT INTO produtos (sku,nome,categoria_id,preco,custo) VALUES ('X4','Prejuízo',1,50,100)")

print("\n⑥ Este passa:")
sql("INSERT INTO produtos (sku,nome,categoria_id,preco,custo,estoque) VALUES ('MO-LG-24','Monitor LG 24',2,1199.00,920.00,31)")

### ⚠️ `NULL` não é zero, nem string vazia

`NULL` significa **"desconhecido"**. E a lógica com desconhecido é traiçoeira:

- `NULL = NULL` → **não é verdadeiro**. Dois desconhecidos não são "iguais".
- `NULL + 10` → `NULL`. Desconhecido mais dez continua desconhecido.
- Para testar, use `IS NULL` / `IS NOT NULL`, **nunca** `= NULL`.

In [ ]:
sql("""
SELECT
    NULL = NULL          AS "NULL = NULL",
    NULL IS NULL         AS "NULL IS NULL",
    NULL + 10            AS "NULL + 10",
    COALESCE(NULL, 0)    AS "COALESCE(NULL,0)",
    'texto' || NULL      AS "concat com NULL"
""")

> 💡 **`COALESCE(a, b, c)`** devolve o primeiro argumento não nulo. É o `.get(chave, padrao)` do SQL e você vai usá-lo o tempo todo em relatórios: `COALESCE(SUM(valor), 0)` evita mostrar `NULL` quando não há linhas.
>
> ⚠️ **Detalhe do `UNIQUE`:** ele permite **vários** `NULL`, porque dois desconhecidos não são considerados iguais. Se a coluna deve ser realmente única e obrigatória, combine `NOT NULL UNIQUE`.

## 7. Chave estrangeira (FOREIGN KEY)

A FK é o que **liga** as tabelas e garante que a ligação faz sentido.

```sql
categoria_id INTEGER NOT NULL REFERENCES categorias(id)
```

Isso significa: *"todo valor aqui precisa existir em `categorias.id`"*. O banco **recusa** um produto de uma categoria inexistente — e recusa apagar uma categoria que tem produtos.

> 🔴 **A pegadinha nº 1 do SQLite:** as chaves estrangeiras vêm **desligadas por padrão**, por compatibilidade com versões antigas. Você precisa executar `PRAGMA foreign_keys = ON` **em toda conexão nova**. Esquecer isso significa ter FKs decorativas que não garantem nada.

In [ ]:
print("Foreign keys ligadas?", CONN.execute("PRAGMA foreign_keys").fetchone()[0])

In [ ]:
# Recriamos 'produtos' agora COM a chave estrangeira declarada
ddl("""
DROP TABLE produtos;

CREATE TABLE produtos (
    id           INTEGER PRIMARY KEY,
    sku          TEXT    NOT NULL UNIQUE,
    nome         TEXT    NOT NULL,
    categoria_id INTEGER NOT NULL REFERENCES categorias(id) ON DELETE RESTRICT,
    preco        REAL    NOT NULL CHECK (preco >= 0),
    custo        REAL    NOT NULL CHECK (custo >= 0),
    estoque      INTEGER NOT NULL DEFAULT 0 CHECK (estoque >= 0),
    ativo        INTEGER NOT NULL DEFAULT 1 CHECK (ativo IN (0,1)),
    CHECK (preco >= custo)
);
""")

sql("""INSERT INTO produtos (sku,nome,categoria_id,preco,custo,estoque) VALUES
    ('NB-DELL-15','Notebook Dell Inspiron 15', 1, 2599.90, 2120.00, 14),
    ('MO-LG-24',  'Monitor LG 24 UltraWide',   2, 1199.00,  920.00, 31),
    ('PE-LOG-M170','Mouse Logitech M170',      3,   89.90,   52.00, 240)""")
sql("SELECT id, sku, nome, categoria_id, preco FROM produtos")

In [ ]:
print("① Produto de uma categoria que não existe:")
sql("INSERT INTO produtos (sku,nome,categoria_id,preco,custo) VALUES ('XX-1','Órfão', 999, 100, 50)")

print("\n② Apagar uma categoria que tem produtos (ON DELETE RESTRICT):")
sql("DELETE FROM categorias WHERE id = 1")

print("\n③ Apagar uma categoria SEM produtos:")
sql("INSERT INTO categorias (nome) VALUES ('Descontinuados')")
sql("DELETE FROM categorias WHERE nome = 'Descontinuados'")

### Ações referenciais: `ON DELETE` e `ON UPDATE`

O que acontece com os "filhos" quando o "pai" é apagado ou alterado?

| Ação | Comportamento | Quando usar |
|------|---------------|-------------|
| `RESTRICT` / `NO ACTION` | **Impede** a operação | Padrão seguro. Categoria com produtos não some. |
| `CASCADE` | Apaga/atualiza os filhos junto | Só quando o filho **não existe sem** o pai: item de pedido sem pedido não faz sentido |
| `SET NULL` | Coloca `NULL` nos filhos | Vendedor saiu da empresa, mas os pedidos permanecem |
| `SET DEFAULT` | Coloca o valor padrão | Raro |

> ⚠️ **`CASCADE` é conveniente e perigoso.** Um `DELETE` numa linha pode disparar uma reação em cadeia que apaga milhares de registros em tabelas que você nem lembrava que existiam. Use apenas em relações de **composição** real (o filho é parte do pai). Na dúvida, use `RESTRICT`.

In [ ]:
# Relação de COMPOSIÇÃO: item de pedido não existe sem o pedido -> CASCADE
ddl("""
CREATE TABLE clientes (
    id            INTEGER PRIMARY KEY,
    nome          TEXT NOT NULL,
    email         TEXT NOT NULL UNIQUE,
    cidade        TEXT NOT NULL,
    uf            TEXT NOT NULL CHECK (length(uf) = 2 AND uf = upper(uf)),
    segmento      TEXT NOT NULL DEFAULT 'varejo'
                       CHECK (segmento IN ('varejo','corporativo')),
    data_cadastro TEXT NOT NULL DEFAULT (date('now'))
);

CREATE TABLE pedidos (
    id          INTEGER PRIMARY KEY,
    cliente_id  INTEGER NOT NULL REFERENCES clientes(id) ON DELETE RESTRICT,
    data_pedido TEXT    NOT NULL,
    status      TEXT    NOT NULL DEFAULT 'pendente'
                        CHECK (status IN ('pago','pendente','cancelado')),
    canal       TEXT    NOT NULL CHECK (canal IN ('site','app','marketplace')),
    frete       REAL    NOT NULL DEFAULT 0 CHECK (frete >= 0)
);

CREATE TABLE itens_pedido (
    id             INTEGER PRIMARY KEY,
    pedido_id      INTEGER NOT NULL REFERENCES pedidos(id)  ON DELETE CASCADE,
    produto_id     INTEGER NOT NULL REFERENCES produtos(id) ON DELETE RESTRICT,
    quantidade     INTEGER NOT NULL CHECK (quantidade > 0),
    preco_unitario REAL    NOT NULL CHECK (preco_unitario >= 0),

    -- o mesmo produto não pode aparecer duas vezes no mesmo pedido
    UNIQUE (pedido_id, produto_id)
);
""")

sql("""INSERT INTO clientes (nome,email,cidade,uf,segmento) VALUES
    ('Maria Souza','maria@email.com','Campinas','SP','varejo'),
    ('João Lima','joao@email.com','São Paulo','SP','corporativo')""")

sql("""INSERT INTO pedidos (cliente_id,data_pedido,status,canal,frete) VALUES
    (1,'2026-07-01','pago','site',9.90),
    (1,'2026-07-08','pago','app',0),
    (2,'2026-07-03','pendente','marketplace',19.90)""")

sql("""INSERT INTO itens_pedido (pedido_id,produto_id,quantidade,preco_unitario) VALUES
    (1,1,2,2599.90),
    (1,3,1,89.90),
    (2,2,1,1199.00),
    (3,1,1,2599.90)""")

sql("SELECT * FROM itens_pedido")

In [ ]:
print("① UF minúscula (o CHECK exige maiúscula):")
sql("INSERT INTO clientes (nome,email,cidade,uf) VALUES ('Teste','t@x.com','Campinas','sp')")

print("\n② Mesmo produto duas vezes no mesmo pedido:")
sql("INSERT INTO itens_pedido (pedido_id,produto_id,quantidade,preco_unitario) VALUES (1,1,5,2599.90)")

print("\n③ Quantidade zero:")
sql("INSERT INTO itens_pedido (pedido_id,produto_id,quantidade,preco_unitario) VALUES (2,3,0,89.90)")

print("\n④ Apagar cliente que tem pedidos (RESTRICT):")
sql("DELETE FROM clientes WHERE id = 1")

In [ ]:
# CASCADE em ação: apagar o pedido leva os itens junto
print("Itens antes:")
sql("SELECT id, pedido_id, produto_id FROM itens_pedido")

sql("DELETE FROM pedidos WHERE id = 3")

print("\nItens depois (o item do pedido 3 sumiu sozinho):")
sql("SELECT id, pedido_id, produto_id FROM itens_pedido")

## 8. `ALTER TABLE` e `DROP TABLE`

```sql
ALTER TABLE tabela ADD COLUMN nova TEXT;
ALTER TABLE tabela RENAME TO novo_nome;
ALTER TABLE tabela RENAME COLUMN antiga TO nova;   -- SQLite 3.25+
ALTER TABLE tabela DROP COLUMN coluna;             -- SQLite 3.35+

DROP TABLE tabela;
DROP TABLE IF EXISTS tabela;    -- não reclama se não existir
```

> ⚠️ **O `ALTER TABLE` do SQLite é limitado.** Você não pode alterar o tipo de uma coluna, nem adicionar uma constraint a uma coluna existente. O procedimento oficial é: criar a tabela nova → copiar os dados → apagar a antiga → renomear. É trabalhoso, e é uma das razões para migrar ao PostgreSQL.
>
> No Módulo 05 você vai conhecer o **Alembic**, que automatiza esse tipo de migração de schema com versionamento.

In [ ]:
sql("ALTER TABLE clientes ADD COLUMN telefone TEXT")
sql("SELECT id, nome, email, telefone FROM clientes")

In [ ]:
# ⚠️ ADD COLUMN com NOT NULL exige um DEFAULT — senão as linhas existentes ficariam inválidas
print("① Sem default:")
sql("ALTER TABLE clientes ADD COLUMN cpf TEXT NOT NULL")

print("\n② Com default:")
sql("ALTER TABLE clientes ADD COLUMN cpf TEXT NOT NULL DEFAULT ''")
sql("SELECT id, nome, cpf FROM clientes")

## 9. Inspecionando o schema

O SQLite guarda o próprio schema numa tabela chamada `sqlite_master`. E oferece comandos `PRAGMA` para inspeção.

In [ ]:
sql("SELECT type, name FROM sqlite_master WHERE type='table' ORDER BY name")

In [ ]:
sql("PRAGMA table_info(pedidos)")

In [ ]:
sql("PRAGMA foreign_key_list(itens_pedido)")

In [ ]:
# O DDL original de qualquer objeto
linha = CONN.execute(
    "SELECT sql FROM sqlite_master WHERE name = 'itens_pedido'"
).fetchone()
print(linha[0])

## 10. O diagrama ER do Atlas

Antes de escrever qualquer `CREATE TABLE`, desenhe. Cinco minutos num papel economizam horas de refatoração.

```
┌──────────────────────┐
│      categorias      │
├──────────────────────┤
│ 🔑 id                │
│    nome        UNIQUE│
│    margem_alvo       │
└──────────┬───────────┘
           │ 1
           │
           │ N
┌──────────┴───────────┐        ┌──────────────────────┐
│      produtos        │        │      clientes        │
├──────────────────────┤        ├──────────────────────┤
│ 🔑 id                │        │ 🔑 id                │
│    sku         UNIQUE│        │    nome              │
│    nome              │        │    email       UNIQUE│
│ 🔗 categoria_id      │        │    cidade            │
│    preco             │        │    uf                │
│    custo             │        │    segmento          │
│    estoque           │        │    data_cadastro     │
│    ativo             │        └──────────┬───────────┘
└──────────┬───────────┘                   │ 1
           │ 1                             │
           │                               │ N
           │ N                  ┌──────────┴───────────┐
           │                    │       pedidos        │
           │                    ├──────────────────────┤
           │                    │ 🔑 id                │
           │                    │ 🔗 cliente_id        │
           │                    │    data_pedido       │
           │                    │    status            │
           │                    │    canal             │
           │                    │    frete             │
           │                    └──────────┬───────────┘
           │                               │ 1
           │                               │
           │            N     ┌────────────┴─────┐  N
           └──────────────────┤  itens_pedido    ├────
                              ├──────────────────┤
                              │ 🔑 id            │
                              │ 🔗 pedido_id     │  CASCADE
                              │ 🔗 produto_id    │  RESTRICT
                              │    quantidade    │
                              │    preco_unitario│
                              │ UNIQUE(pedido,produto)
                              └──────────────────┘

🔑 chave primária    🔗 chave estrangeira
```

### Lendo as cardinalidades

| Relação | Leitura |
|---------|---------|
| `categorias 1 ── N produtos` | Uma categoria tem muitos produtos; cada produto tem uma categoria |
| `clientes 1 ── N pedidos` | Um cliente faz muitos pedidos; cada pedido é de um cliente |
| `pedidos 1 ── N itens_pedido` | Um pedido tem muitos itens |
| `produtos 1 ── N itens_pedido` | Um produto aparece em muitos itens |

### A tabela de junção

`pedidos` e `produtos` têm uma relação **N para N**: um pedido tem vários produtos, e um produto está em vários pedidos.

Bancos relacionais **não representam N-N diretamente**. A solução é uma tabela intermediária — aqui, `itens_pedido`.

E ela ganha um bônus importante: pode ter **atributos próprios**. `quantidade` e `preco_unitario` não pertencem nem ao pedido nem ao produto — pertencem à **combinação** dos dois.

> 💡 **Por que guardar `preco_unitario` no item, se o produto já tem `preco`?** Porque o preço do produto **muda com o tempo**. Se você calculasse o faturamento de julho usando o preço de hoje, os números históricos mudariam sozinhos toda vez que alguém reajustasse o catálogo. O `preco_unitario` congela o valor **praticado naquela venda**. Isso se chama *snapshot* de dado histórico e é um padrão fundamental em sistemas transacionais.

## 🔧 Prática guiada — Populando e conferindo

In [ ]:
# Mais dados para o banco ficar interessante
sql("""INSERT INTO categorias (nome, margem_alvo) VALUES
    ('Armazenamento', 0.30), ('Redes', 0.28), ('Áudio', 0.35)""")

sql("""INSERT INTO produtos (sku,nome,categoria_id,preco,custo,estoque) VALUES
    ('NB-ACER-N5','Notebook Acer Nitro 5',      1, 3299.00, 2780.00,  7),
    ('MO-SAM-ODY','Monitor Samsung Odyssey 27', 2, 1849.00, 1420.00, 12),
    ('PE-RED-K552','Teclado Redragon K552',     3,  249.00,  150.00, 64),
    ('AR-SSD-1TB','SSD NVMe 1TB Kingston',      4,  489.00,  360.00, 52),
    ('RE-TPL-AX55','Roteador TP-Link AX55',     5,  699.00,  505.00, 26),
    ('AU-HYP-CL2','Headset HyperX Cloud II',    6,  399.00,  255.00, 29)""")

sql("""INSERT INTO clientes (nome,email,cidade,uf,segmento) VALUES
    ('Ana Costa','ana@email.com','Campinas','SP','varejo'),
    ('Bruno Rocha','bruno@email.com','Ribeirão Preto','SP','corporativo'),
    ('Carla Dias','carla@email.com','Curitiba','PR','varejo')""")

sql("SELECT COUNT(*) AS categorias FROM categorias")

In [ ]:
# Contagem de todas as tabelas
sql("""
SELECT 'categorias'   AS tabela, COUNT(*) AS linhas FROM categorias
UNION ALL SELECT 'produtos',     COUNT(*) FROM produtos
UNION ALL SELECT 'clientes',     COUNT(*) FROM clientes
UNION ALL SELECT 'pedidos',      COUNT(*) FROM pedidos
UNION ALL SELECT 'itens_pedido', COUNT(*) FROM itens_pedido
""")

In [ ]:
# Uma primeira consulta juntando tudo (JOIN é a aula 03_03 — aqui é só um aperitivo)
sql("""
SELECT
    p.id                                  AS pedido,
    c.nome                                AS cliente,
    c.cidade,
    pr.nome                               AS produto,
    cat.nome                              AS categoria,
    i.quantidade                          AS qtd,
    i.preco_unitario                      AS preco,
    ROUND(i.quantidade * i.preco_unitario, 2) AS total
FROM itens_pedido i
JOIN pedidos    p   ON p.id   = i.pedido_id
JOIN clientes   c   ON c.id   = p.cliente_id
JOIN produtos   pr  ON pr.id  = i.produto_id
JOIN categorias cat ON cat.id = pr.categoria_id
WHERE p.status = 'pago'
ORDER BY total DESC
""")

> 💭 **Compare com o Módulo 01.** Aquele mesmo relatório exigiu ~40 linhas de Python: laços, dicionários, `defaultdict`. Aqui são 12 linhas declarativas — e você diz **o que quer**, não **como buscar**. O banco decide o melhor caminho de execução. Isso é o poder do SQL.

## 📝 Exercícios rápidos

**E1.** Crie uma tabela `fornecedores` com: `id` (PK), `cnpj` (único, obrigatório), `razao_social` (obrigatório), `uf` (2 letras maiúsculas), `ativo` (0 ou 1, padrão 1). Teste cada constraint tentando violá-la.

**E2.** Adicione `fornecedor_id` à tabela `produtos` como FK opcional (pode ser `NULL`), com `ON DELETE SET NULL`. Explique por que `SET NULL` faz sentido aqui e `CASCADE` não faria.

**E3.** Crie `enderecos_entrega` com FK para `clientes` e `ON DELETE CASCADE`. Um cliente pode ter vários endereços. Adicione um `CHECK` que garanta CEP com 8 dígitos.

**E4.** Crie `historico_precos` (produto_id, data_inicio, preco) com PK composta `(produto_id, data_inicio)`. Insira três reajustes de um produto e explique por que esta tabela existe.

**E5.** Descubra, usando `PRAGMA`, quantas colunas tem `itens_pedido`, quais são obrigatórias e quais são as FKs.

**E6.** Tente inserir um pedido com `status = 'entregue'`. Depois altere o `CHECK` para aceitar esse status novo. (Dica: o SQLite não permite alterar um `CHECK` — pesquise o procedimento de recriação da tabela.)

**E7.** Explique em uma célula markdown: por que `itens_pedido` guarda `preco_unitario` se `produtos` já tem `preco`?

In [ ]:
# E1

In [ ]:
# E2

In [ ]:
# E3

In [ ]:
# E4

In [ ]:
# E5

In [ ]:
# E6

<!-- E7 — sua resposta aqui -->

## 📋 Cola de referência

```sql
-- ── Criar tabela ──
CREATE TABLE nome (
    id      INTEGER PRIMARY KEY,           -- autoincrementa no SQLite
    codigo  TEXT    NOT NULL UNIQUE,
    valor   REAL    NOT NULL CHECK (valor >= 0),
    ativo   INTEGER NOT NULL DEFAULT 1 CHECK (ativo IN (0,1)),
    criado  TEXT    NOT NULL DEFAULT (date('now')),
    pai_id  INTEGER NOT NULL REFERENCES pai(id) ON DELETE RESTRICT,
    UNIQUE (codigo, pai_id),               -- constraint de tabela
    CHECK (valor >= 0)
);

CREATE TABLE composta (
    a INTEGER, b TEXT,
    PRIMARY KEY (a, b)
);

-- ── Alterar ──
ALTER TABLE t ADD COLUMN nova TEXT;
ALTER TABLE t ADD COLUMN nova TEXT NOT NULL DEFAULT '';
ALTER TABLE t RENAME TO novo_nome;
ALTER TABLE t RENAME COLUMN a TO b;
ALTER TABLE t DROP COLUMN c;

-- ── Remover ──
DROP TABLE t;
DROP TABLE IF EXISTS t;

-- ── Ações referenciais ──
ON DELETE RESTRICT     -- impede (padrão seguro)
ON DELETE CASCADE      -- apaga os filhos (só em composição real)
ON DELETE SET NULL     -- desvincula
ON UPDATE CASCADE      -- propaga a mudança de chave

-- ── Inspeção ──
SELECT type, name FROM sqlite_master WHERE type='table';
SELECT sql  FROM sqlite_master WHERE name='tabela';
PRAGMA table_info(tabela);
PRAGMA foreign_key_list(tabela);
PRAGMA index_list(tabela);

-- ── Configuração (SEMPRE em conexão nova) ──
PRAGMA foreign_keys = ON;
```

```python
# ── Python ──
import sqlite3
conn = sqlite3.connect("aurora.db")        # ou ":memory:"
conn.execute("PRAGMA foreign_keys = ON")   # OBRIGATÓRIO
conn.execute("SELECT ...").fetchall()
conn.executescript("...vários comandos...")
conn.commit()
conn.close()
```

## ✅ Checklist de saída

- [ ] Explico as três anomalias do arquivo plano
- [ ] Sei quando separar dados em uma tabela nova ("estou copiando o mesmo valor?")
- [ ] Entendo 1FN, 2FN e 3FN no nível da intuição
- [ ] Conecto ao SQLite pelo Python e sei que `:memory:` some ao reiniciar
- [ ] **Sempre** executo `PRAGMA foreign_keys = ON`
- [ ] Sei que o SQLite tem afinidade de tipo, e que `STRICT` resolve
- [ ] Sei como o SQLite guarda datas e booleanos
- [ ] Escolho chave artificial como PK e protejo as naturais com `UNIQUE`
- [ ] Uso `NOT NULL`, `UNIQUE`, `CHECK` e `DEFAULT` deliberadamente
- [ ] Escolho entre `RESTRICT`, `CASCADE` e `SET NULL` com critério
- [ ] Sei que `NULL = NULL` não é verdadeiro e uso `IS NULL`
- [ ] Sei por que uma tabela de junção resolve N-N e pode ter atributos próprios
- [ ] Entendo por que `preco_unitario` é congelado no item do pedido

---

### ➡️ Próxima aula

**`03_02_Consultas_Basicas.ipynb`** — `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY` e `HAVING`. Onde você começa a fazer perguntas ao banco.